# Setup

In [ ]:
%run notebook_setup.py

import pandas as pd
import numpy as np
from scipy.stats import trim_mean
import polars as pl
import pandas as pd
import logging
import gc
import random
import datetime as dt

import matplotlib.pyplot as plt
import seaborn as sns

from src.config.dir_config import OUTPUT_PATH_DEMAND_SUMMARY
from src.config.bigquery_config import CREDENTIALS_GBQ, PROJECT_ID_GBQ
from src.utils.read_data import read_data
from src.utils.setup_logging import setup_logging
from src.utils.create_week_date import add_week_start_date
from src.utils.export_as_excel import export_dataframes_as_tables


import logging

setup_logging()

# Functions

In [ ]:
def add_real_sales_sv(df):
    def var_forward_sum(g):
        g = g.sort_values("week_number")
        sales = g["weekly_sales"].fillna(0).to_numpy(int)
        weeks = g["semana_vta"].fillna(1).astype(int).to_numpy()
        n = len(sales)

        s = np.cumsum(sales)
        i = np.arange(n)
        right = np.minimum(i + weeks, n)
        s_pad = np.concatenate(([0.0], s))
        
        return pd.Series(s_pad[right] - s_pad[i], index=g.index)

    df["real_sales"] = (
    df
    .groupby(["cod_sucursal", "cod_producto", "cod_talla"], group_keys=False)[["week_number", "weekly_sales", "semana_vta"]]
    .apply(var_forward_sum)
    .astype("int16")
    )

    return df

# Data import

In [ ]:
data_genex = pd.read_parquet('../data/processed/genex_data_processed.parquet')
demand_summary = pd.read_parquet('../data/processed/demand_summary.parquet')
data_sales = pd.read_parquet('../data/processed/weekly_sales_by_season/Invierno/weekly_sales_Invierno_2025_processed.parquet')
classifications = pd.read_excel('../data/external/Consolidado clasificaciones modelo BI.xlsx')
transfers = pd.read_parquet('../data/processed/trf_data_processed.parquet')

# Data processing

In [ ]:
demand_summary = demand_summary[demand_summary['cod_sucursal'] != 767]
demand_summary = demand_summary[demand_summary['nombre_depto'] != 'Bolsas y bolsos']
demand_summary = demand_summary[demand_summary['nombre_depto'] != 'Miscelaneos']
demand_summary = demand_summary[demand_summary['nombre_temporada'] == 'Invierno']
demand_summary = demand_summary[demand_summary['ano_temporada'] == '2025']

In [ ]:
demand_info = demand_summary[['cod_producto', 'cod_talla', 'cod_sucursal',
                              'nombre_sucursal',
                              'nombre_temporada','ano_temporada','nombre_depto','nombre_linea','nom_talla',
                              'demand_type']].copy()

In [ ]:
transfers_pivot = transfers.pivot_table(
    index=['cod_sucursal','cod_producto','cod_talla','cod_ano_comercial','cod_semana'],
    observed=True,
    columns="nombre_razon_group",
    values= 'cantidad_des',
    aggfunc="sum",
    fill_value=0
).reset_index()

transfers_pivot.columns.name = None

In [ ]:
transfers_pivot_summary = transfers.pivot_table(
    index=['cod_sucursal','cod_producto','cod_talla'],
    observed=True,
    columns="nombre_razon_group",
    values= 'cantidad_des',
    aggfunc="sum",
    fill_value=0
).reset_index()

transfers_pivot_summary.columns.name = None

In [ ]:
demand_summary = demand_summary.merge(
    transfers_pivot_summary,
    on = ['cod_sucursal','cod_producto','cod_talla'],
    how = 'left'
)

demand_summary[["PREDISTRIBUIDA",
        "REPOSICION AUTOMATIC",
        "CARGA MANUAL",
        "OTRAS"]] = demand_summary[["PREDISTRIBUIDA",
                                    "REPOSICION AUTOMATIC",
                                    "CARGA MANUAL",
                                    "OTRAS"]].fillna(0)

## *A) Proccessing data_all*

In [ ]:
data_all = data_sales.merge(data_genex.drop(columns=['cod_talla']),
                            on=['cod_producto','cod_sku', 'cod_sucursal', 'cod_ano_comercial','cod_semana'],
                                how='left')

data_all = data_all.merge(demand_info,
                            on=['cod_producto','cod_talla', 'cod_sucursal'],
                            how='left')

data_all = data_all.merge(classifications,
                            on=['nombre_temporada','cod_sucursal','nombre_depto', 'nombre_linea'],
                            how='left')

data_all = data_all.merge(transfers_pivot,
                            on=['cod_producto','cod_talla', 'cod_sucursal', 'cod_ano_comercial','cod_semana'],
                                how='left')

del demand_info, classifications, data_sales
gc.collect()

In [ ]:
data_all = add_week_start_date(data_all,
                               year_col='cod_ano_comercial',
                               week_col='cod_semana')

In [ ]:
data_all['clasificacion'] = pd.Categorical(data_all['clasificacion'],
                                           categories=['AA','A','B','C'],
                                           ordered=True)

In [ ]:
data_all[["PREDISTRIBUIDA",
        "REPOSICION AUTOMATIC",
        "CARGA MANUAL",
        "OTRAS"]] = data_all[["PREDISTRIBUIDA",
                                    "REPOSICION AUTOMATIC",
                                    "CARGA MANUAL",
                                    "OTRAS"]].fillna(0)

In [ ]:
data_all = data_all[data_all['cod_sucursal'] != 767]
data_all = data_all[data_all['nombre_depto'] != 'Bolsas y bolsos']
data_all = data_all[data_all['nombre_depto'] != 'Miscelaneos']
data_all = data_all[data_all['nombre_temporada'] == 'Invierno']
data_all = data_all[data_all['ano_temporada'] == '2025']

In [ ]:
dict_factor_l_dias = {
    2: 28/7,
    3: 28/14,
    4: 28/21,
}

dict_factor_l_dias = {
    2: 28/7,   # 4.0
    3: 28/14,  # 2.0
    4: 28/21,  # 1.333...
}

# Condiciones
mask_fecha = data_all['week_start_date'].dt.date == dt.date(2025, 5, 5)
mask_factores = data_all['week_number'].isin(dict_factor_l_dias.keys())

# Calculamos el nuevo vta_promedio en dos pasos
data_all['vta_promedio'] = np.where(
    mask_fecha & mask_factores,
    data_all['mean_sales_past_4_weeks'] * data_all['week_number'].map(dict_factor_l_dias),
    np.where(
        mask_fecha & ~mask_factores,
        data_all['mean_sales_past_4_weeks'],
        data_all['vta_promedio']
    )
)

# Redondeo al final (una sola vez)
data_all['vta_promedio'] = data_all['vta_promedio'].round(3)

In [ ]:
data_all['factor_l_dias'] = np.where(
    data_all['mean_sales_past_4_weeks'] == 0,
    np.nan,
    (data_all['vta_promedio'] / data_all['mean_sales_past_4_weeks']).round(3)
)

In [ ]:
#cod_sku = 641769518
#cod_sucursal = 17

# Filtrar los datos base
data_sample = data_all[data_all['id'] == '204641690119'].copy()
#data_sample = data_sample[data_sample['cod_sucursal'] == cod_sucursal].reset_index(drop=True)


# Ver columnas de interés
data_sample[[
    'cod_sucursal','cod_producto','cod_sku','week_number','week_start_date',
    'can_final','repo_x_dda','weekly_sales', 'stock_start_week',
    'mean_sales_past_4_weeks','vta_promedio','factor_l_dias','factor','semana_vta',
]]

In [ ]:
data_all = add_real_sales_sv(data_all)

# Export

In [ ]:
data_all.to_parquet('../sandbox/data_genex_venta_transfer.parquet')

In [ ]:
demand_summary.to_parquet('../sandbox/demand_summary.parquet')

---

# EDA

In [ ]:
data_all = pd.read_parquet('../sandbox/data_genex_venta_transfer.parquet')

## Sample product

In [ ]:
cod_sku = 641769518
cod_sucursal = 17

# Filtrar los datos base
data_sample = data_all[data_all['cod_sku'] == cod_sku].copy()
data_sample = data_sample[data_sample['cod_sucursal'] == cod_sucursal].reset_index(drop=True)


# Ver columnas de interés
data_sample[[
    'cod_sucursal','cod_producto','cod_sku','week_number','week_start_date',
    'can_final','repo_x_dda','weekly_sales', 'stock_start_week',
    'mean_sales_past_4_weeks','vta_promedio','factor_l_dias','factor','semana_vta',
]]